# Language Identification using IndicCorpV2

- Dataset: 1000 sentences per language, 23 languages
- Features: Custom TF-IDF (word unigrams, bigrams + char 2/3/4-grams)
- Model: Custom Logistic Regression (no sklearn)
- Evaluation: Custom Macro-F1 (no sklearn)

## Step 1: Install Dependencies

In [1]:
!pip install datasets huggingface_hub numpy

## Step 2: Import Libraries

In [2]:
import sys
import re
import math
import random
import numpy as np
from collections import Counter
from datasets import load_dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print('Imports OK.')


Imports OK.


## Step 3: Define All 23 Language Splits

In [3]:
LANGUAGE_SPLITS = [
    "asm_Beng", "ben_Beng", "brx_Deva", "doi_Deva", "gom_Deva",
    "guj_Gujr", "hin_Deva", "kan_Knda", "kas_Arab", "mai_Deva",
    "mal_Mlym", "mar_Deva", "mni_Mtei", "npi_Deva", "ory_Orya",
    "pan_Guru", "san_Deva", "snd_Deva", "tam_Taml", "tel_Telu",
    "urd_Arab", "khasi", "santhali",
]
print(f"Total languages: {len(LANGUAGE_SPLITS)}")
for lang in LANGUAGE_SPLITS:
    print(f"  {lang}")


Total languages: 23
  asm_Beng
  ben_Beng
  brx_Deva
  doi_Deva
  gom_Deva
  guj_Gujr
  hin_Deva
  kan_Knda
  kas_Arab
  mai_Deva
  mal_Mlym
  mar_Deva
  mni_Mtei
  npi_Deva
  ory_Orya
  pan_Guru
  san_Deva
  snd_Deva
  tam_Taml
  tel_Telu
  urd_Arab
  khasi
  santhali


## Step 4: Sentence Tokenizer

Rule-based tokenizer handling Devanagari danda (\u0964), double danda (\u0965), and standard punctuation.

In [4]:
def tokenize_sentences(text):
    """
    Tokenize a paragraph into sentences.
    Handles Devanagari danda (\u0964), double danda (\u0965),
    and standard sentence-ending punctuation (. ! ?).
    """
    if not text or not text.strip():
        return []

    # Split after sentence-ending punctuation followed by whitespace
    # \u0964 = Devanagari danda, \u0965 = double danda
    pattern = r'(?<=[\u0964\u0965.!?])\s+|(?<=[\u0964\u0965.!?])$'
    sentences = re.split(pattern, text.strip())

    cleaned = []
    for s in sentences:
        s = s.strip()
        if len(s) >= 3:
            cleaned.append(s)
    return cleaned


# Test with Hindi
test_hi = "\u092f\u0939 \u090f\u0915 \u0935\u093e\u0915\u094d\u092f \u0939\u0948\u0964 \u092f\u0939 \u0926\u0942\u0938\u0930\u093e \u0935\u093e\u0915\u094d\u092f \u0939\u0948\u0964"
print("Hindi:", tokenize_sentences(test_hi))
test_en = "This is one. This is two! Is this three?"
print("English:", tokenize_sentences(test_en))


Hindi: ['यह एक वाक्य है।', 'यह दूसरा वाक्य है।']
English: ['This is one.', 'This is two!', 'Is this three?']


## Step 5: Load Data from IndicCorpV2

Stream each language split, tokenize paragraphs into sentences, and collect exactly 1000 sentences per language.

**Data is cached** to `dataset_sentences.pkl` next to this notebook. On subsequent runs the cache is loaded instantly — no re-downloading needed.

In [5]:
import os, pickle
import urllib.request

TARGET_SENTENCES = random.randrange(900,1000)
MAX_PARAGRAPHS   = 5000   # safety cap per language

CACHE_FILE = os.path.join(os.getcwd(), 'dataset_sentences.pkl')

LANGUAGE_TO_FILE = {
    'asm_Beng': 'as.txt', 'ben_Beng': 'bn.txt', 'brx_Deva': 'bd.txt',
    'doi_Deva': 'dg.txt', 'gom_Deva': 'gom.txt', 'guj_Gujr': 'gu.txt',
    'hin_Deva': 'hi-1.txt', 'kan_Knda': 'kn.txt', 'kas_Arab': 'ks.txt',
    'mai_Deva': 'mai.txt', 'mal_Mlym': 'ml.txt', 'mar_Deva': 'mr.txt',
    'mni_Mtei': 'mni.txt', 'npi_Deva': 'ne.txt', 'ory_Orya': 'or.txt',
    'pan_Guru': 'pa.txt', 'san_Deva': 'sa.txt', 'snd_Deva': 'sd.txt',
    'tam_Taml': 'ta.txt', 'tel_Telu': 'te.txt', 'urd_Arab': 'ur.txt',
    'khasi': 'kha.txt', 'santhali': 'sat.txt'
}

if os.path.exists(CACHE_FILE):
    print(f'Cache found -> {CACHE_FILE}')
    with open(CACHE_FILE, 'rb') as f:
        dataset_sentences = pickle.load(f)
    print('Loaded from cache. Summary:')
    for lang, sents in dataset_sentences.items():
        print(f'  {lang}: {len(sents)} sentences')

else:
    print('No cache found. Streaming manually from HuggingFace via urllib...\n')
    dataset_sentences = {}

    for lang in LANGUAGE_SPLITS:
        print(f'Loading {lang}...', end=' ', flush=True)
        collected = []
        fname = LANGUAGE_TO_FILE.get(lang)

        if not fname:
            print(f'ERROR: No file mapping for {lang}')
            dataset_sentences[lang] = []
            continue

        url = f"https://huggingface.co/datasets/ai4bharat/IndicCorpV2/resolve/main/data/{fname}"

        try:
            req = urllib.request.Request(url)
            with urllib.request.urlopen(req) as response:
                para_count = 0
                for line in response:
                    if len(collected) >= TARGET_SENTENCES or para_count >= MAX_PARAGRAPHS:
                        break
                    decoded_line = line.decode('utf-8').strip()
                    if decoded_line:
                        for s in tokenize_sentences(decoded_line):
                            if len(collected) >= TARGET_SENTENCES:
                                break
                            collected.append(s)
                    para_count += 1
            dataset_sentences[lang] = collected
            print(f'{len(collected)} sentences from {para_count} paragraphs.')
        except Exception as ex:
            print(f'ERROR: {ex}')
            dataset_sentences[lang] = []

    with open(CACHE_FILE, 'wb') as f:
        pickle.dump(dataset_sentences, f)
    print(f'\nSentence data cached to: {CACHE_FILE}')

print(f'\nTotal languages loaded  : {len(dataset_sentences)}')
langs_with_data = {l: len(s) for l, s in dataset_sentences.items() if len(s) > 0}
print(f'Languages with sentences: {len(langs_with_data)}/{len(dataset_sentences)}')
if len(langs_with_data) < len(dataset_sentences):
    missing = [l for l, s in dataset_sentences.items() if len(s) == 0]
    print(f'WARNING - missing data  : {missing}')


No cache found. Streaming manually from HuggingFace via urllib...

Loading asm_Beng... 981 sentences from 909 paragraphs.
Loading ben_Beng... 981 sentences from 537 paragraphs.
Loading brx_Deva... 981 sentences from 1147 paragraphs.
Loading doi_Deva... 981 sentences from 1397 paragraphs.
Loading gom_Deva... 981 sentences from 881 paragraphs.
Loading guj_Gujr... 981 sentences from 711 paragraphs.
Loading hin_Deva... 981 sentences from 609 paragraphs.
Loading kan_Knda... 981 sentences from 839 paragraphs.
Loading kas_Arab... 981 sentences from 1909 paragraphs.
Loading mai_Deva... 981 sentences from 855 paragraphs.
Loading mal_Mlym... 981 sentences from 529 paragraphs.
Loading mar_Deva... 981 sentences from 625 paragraphs.
Loading mni_Mtei... 981 sentences from 1819 paragraphs.
Loading npi_Deva... 981 sentences from 813 paragraphs.
Loading ory_Orya... 981 sentences from 1413 paragraphs.
Loading pan_Guru... 981 sentences from 691 paragraphs.
Loading san_Deva... 981 sentences from 623 parag

## Step 6: Stratified Split (80:10:10)

In [6]:
def stratified_split(dataset_sentences, train_r=0.8, val_r=0.1, test_r=0.1, seed=42):
    """Stratified split: equal per-language proportion in each set."""
    assert abs(train_r + val_r + test_r - 1.0) < 1e-9, 'ratios must sum to 1'
    rng = random.Random(seed)
    train_texts, train_labels = [], []
    val_texts,   val_labels   = [], []
    test_texts,  test_labels  = [], []
    for lang, sents in dataset_sentences.items():
        s = sents[:]
        rng.shuffle(s)
        n = len(s)
        nt = int(n * train_r)
        nv = int(n * val_r)
        train_texts.extend(s[:nt]);        train_labels.extend([lang]*nt)
        val_texts.extend(s[nt:nt+nv]);     val_labels.extend([lang]*nv)
        test_texts.extend(s[nt+nv:]);      test_labels.extend([lang]*(n-nt-nv))
    return train_texts, train_labels, val_texts, val_labels, test_texts, test_labels


train_texts, train_labels, val_texts, val_labels, test_texts, test_labels = \
    stratified_split(dataset_sentences, seed=SEED)

print(f"Train: {len(train_texts)} | Val: {len(val_texts)} | Test: {len(test_texts)}")

train_dist = Counter(train_labels)
val_dist   = Counter(val_labels)
test_dist  = Counter(test_labels)
print()
print(f"{'Language':<15} {'Train':>6} {'Val':>6} {'Test':>6}")
print('-' * 38)
for lang in LANGUAGE_SPLITS:
    print(f"{lang:<15} {train_dist.get(lang,0):>6} {val_dist.get(lang,0):>6} {test_dist.get(lang,0):>6}")


Train: 18032 | Val: 2254 | Test: 2277

Language         Train    Val   Test
--------------------------------------
asm_Beng           784     98     99
ben_Beng           784     98     99
brx_Deva           784     98     99
doi_Deva           784     98     99
gom_Deva           784     98     99
guj_Gujr           784     98     99
hin_Deva           784     98     99
kan_Knda           784     98     99
kas_Arab           784     98     99
mai_Deva           784     98     99
mal_Mlym           784     98     99
mar_Deva           784     98     99
mni_Mtei           784     98     99
npi_Deva           784     98     99
ory_Orya           784     98     99
pan_Guru           784     98     99
san_Deva           784     98     99
snd_Deva           784     98     99
tam_Taml           784     98     99
tel_Telu           784     98     99
urd_Arab           784     98     99
khasi              784     98     99
santhali           784     98     99


## Step 7: Encode Labels

In [7]:
unique_labels = sorted(set(train_labels))
label2idx = {lbl: i for i, lbl in enumerate(unique_labels)}
idx2label = {i: lbl for lbl, i in label2idx.items()}
NUM_CLASSES = len(unique_labels)

print(f"Number of classes: {NUM_CLASSES}")
for lbl, idx in label2idx.items():
    print(f"  {lbl} -> {idx}")

y_train = np.array([label2idx[l] for l in train_labels])
y_val   = np.array([label2idx[l] for l in val_labels])
y_test  = np.array([label2idx[l] for l in test_labels])


Number of classes: 23
  asm_Beng -> 0
  ben_Beng -> 1
  brx_Deva -> 2
  doi_Deva -> 3
  gom_Deva -> 4
  guj_Gujr -> 5
  hin_Deva -> 6
  kan_Knda -> 7
  kas_Arab -> 8
  khasi -> 9
  mai_Deva -> 10
  mal_Mlym -> 11
  mar_Deva -> 12
  mni_Mtei -> 13
  npi_Deva -> 14
  ory_Orya -> 15
  pan_Guru -> 16
  san_Deva -> 17
  santhali -> 18
  snd_Deva -> 19
  tam_Taml -> 20
  tel_Telu -> 21
  urd_Arab -> 22


## Step 8: Custom TF-IDF Vectorizer

**No sklearn used.** Features:
- Word unigrams (`W:<token>`)
- Word bigrams (`W:<t1> <t2>`)
- Char 2-grams (`C2:<xy>`)
- Char 3-grams (`C3:<xyz>`)
- Char 4-grams (`C4:<xyzw>`)

TF(t,d) = 1 + log(1 + count(t,d))  
IDF(t) = log((1+N)/(1+df(t))) + 1  (smooth IDF)

In [8]:
class CustomTFIDFVectorizer:
    """
    Custom TF-IDF vectorizer with word and character n-grams.
    No sklearn classes used.

    Word n-grams: unigrams (n=1) and bigrams (n=2)
    Char n-grams: 2-grams, 3-grams, 4-grams

    Formulas used:
    TF(t,d)  = 1 + log(1 + raw_freq)
    IDF(t)   = 1 + log((N + 1) / (df + 1))
    Score    = TF * IDF (No normalization)
    """

    def __init__(self, max_features=50000, min_df=2):
        self.max_features = max_features
        self.min_df = min_df
        self.vocabulary_ = {}   # term -> index
        self.idf_values_ = {}   # term -> idf value
        self.vocab_list_ = []   # ordered terms

    # ---- Feature extraction ----

    def _word_ngrams(self, text, n):
        tokens = text.split()
        if len(tokens) < n:
            return []
        return ['W:' + ' '.join(tokens[i:i+n]) for i in range(len(tokens)-n+1)]

    def _char_ngrams(self, text, n):
        text = text.strip()
        if len(text) < n:
            return []
        return [f'C{n}:' + text[i:i+n] for i in range(len(text)-n+1)]

    def _extract_features(self, text):
        feats = []
        feats.extend(self._word_ngrams(text, 1))   # word unigrams
        feats.extend(self._word_ngrams(text, 2))   # word bigrams
        feats.extend(self._char_ngrams(text, 2))   # char 2-grams
        feats.extend(self._char_ngrams(text, 3))   # char 3-grams
        feats.extend(self._char_ngrams(text, 4))   # char 4-grams
        return feats

    # ---- TF computation ----

    def _compute_tf(self, features):
        """TF = 1 + log(1 + raw_freq)"""
        if not features:
            return {}
        counts = Counter(features)
        return {t: 1 + math.log(1 + c) for t, c in counts.items()}

    # ---- Fit ----

    def fit(self, texts):
        """Build vocabulary and compute IDF from training texts."""
        print('Extracting features from training data...')
        N = len(texts)
        all_features = []
        for i, text in enumerate(texts):
            all_features.append(self._extract_features(text))
            if (i+1) % 2000 == 0:
                print(f'  {i+1}/{N} docs...')

        # Document frequency
        print('Computing IDF...')
        df = Counter()
        for feats in all_features:
            for t in set(feats):
                df[t] += 1

        # IDF = 1 + log((N + 1) / (df + 1))
        idf_all = {
            t: 1 + math.log((N + 1) / (freq + 1))
            for t, freq in df.items()
            if freq >= self.min_df
        }
        print(f'Vocab before max_features filter: {len(idf_all)}')

        # Select top max_features by df
        if self.max_features and len(idf_all) > self.max_features:
            top_terms = sorted(idf_all, key=lambda t: df[t], reverse=True)[:self.max_features]
        else:
            top_terms = list(idf_all.keys())

        self.vocab_list_ = sorted(top_terms)
        self.vocabulary_ = {t: i for i, t in enumerate(self.vocab_list_)}
        self.idf_values_ = {t: idf_all[t] for t in self.vocab_list_}
        print(f'Final vocabulary size: {len(self.vocabulary_)}')
        return self

    # ---- Transform ----

    def _transform_one(self, text):
        feats = self._extract_features(text)
        tf = self._compute_tf(feats)
        tfidf = {
            self.vocabulary_[t]: tf[t] * self.idf_values_[t]
            for t in tf if t in self.vocabulary_
        }
        return tfidf

    def transform(self, texts):
        """Transform texts to TF-IDF matrix (N, vocab_size)."""
        V = len(self.vocabulary_)
        N = len(texts)
        X = np.zeros((N, V), dtype=np.float32)
        for i, text in enumerate(texts):
            for idx, val in self._transform_one(text).items():
                X[i, idx] = val
            if (i+1) % 2000 == 0:
                print(f'  {i+1}/{N} vectorized...')
        return X

    def fit_transform(self, texts):
        self.fit(texts)
        print('Transforming training data...')
        return self.transform(texts)


print('CustomTFIDFVectorizer defined.')


CustomTFIDFVectorizer defined.


## Step 9: Fit TF-IDF and Transform All Splits

In [9]:
vectorizer = CustomTFIDFVectorizer(max_features=50000, min_df=2)

print('=' * 60)
print('Fitting on training data...')
print('=' * 60)
X_train = vectorizer.fit_transform(train_texts)

print('\nTransforming validation data...')
X_val = vectorizer.transform(val_texts)

print('\nTransforming test data...')
X_test = vectorizer.transform(test_texts)

print(f'X_train: {X_train.shape}')
print(f'X_val  : {X_val.shape}')
print(f'X_test : {X_test.shape}')


Fitting on training data...
Extracting features from training data...
  2000/18032 docs...
  4000/18032 docs...
  6000/18032 docs...
  8000/18032 docs...
  10000/18032 docs...
  12000/18032 docs...
  14000/18032 docs...
  16000/18032 docs...
  18000/18032 docs...
Computing IDF...
Vocab before max_features filter: 353868
Final vocabulary size: 50000
Transforming training data...
  2000/18032 vectorized...
  4000/18032 vectorized...
  6000/18032 vectorized...
  8000/18032 vectorized...
  10000/18032 vectorized...
  12000/18032 vectorized...
  14000/18032 vectorized...
  16000/18032 vectorized...
  18000/18032 vectorized...

Transforming validation data...
  2000/2254 vectorized...

Transforming test data...
  2000/2277 vectorized...
X_train: (18032, 50000)
X_val  : (2254, 50000)
X_test : (2277, 50000)


## Step 10: Custom Logistic Regression Classifier

**No sklearn used.** Multi-class logistic regression with:
- Softmax activation
- Cross-entropy loss
- L2 regularization
- Mini-batch SGD optimizer
- Xavier weight initialization

In [10]:
class CustomLogisticRegression:
    """
    Multi-class Logistic Regression (no sklearn).
    Forward:  z = X @ W + b
              P = softmax(z)  (numerically stable)
    Loss:     -mean(log P[correct_class]) + 0.5*l2*||W||^2
    Backward: gradient of loss w.r.t. W, b
    Optimizer: mini-batch SGD
    """

    def __init__(self, num_classes, learning_rate=0.5, num_epochs=30,
                 batch_size=256, l2_reg=1e-4, seed=42):
        self.num_classes   = num_classes
        self.learning_rate = learning_rate
        self.num_epochs    = num_epochs
        self.batch_size    = batch_size
        self.l2_reg        = l2_reg
        self.seed          = seed
        self.W             = None   # (F, C)
        self.b             = None   # (C,)
        self.train_losses  = []
        self.val_losses    = []

    def _softmax(self, Z):
        """Numerically stable softmax. Z: (N,C) -> P: (N,C)"""
        Z_s = Z - np.max(Z, axis=1, keepdims=True)
        exp_Z = np.exp(Z_s)
        return exp_Z / np.sum(exp_Z, axis=1, keepdims=True)

    def _loss(self, probs, y):
        """Cross-entropy + L2 regularization."""
        N = len(y)
        log_p = -np.log(np.clip(probs[np.arange(N), y], 1e-12, 1.0))
        return float(np.mean(log_p) + 0.5 * self.l2_reg * np.sum(self.W**2))

    def _gradients(self, X_b, y_b):
        """Compute gradients of loss w.r.t. W and b."""
        N = X_b.shape[0]
        Z  = X_b @ self.W + self.b          # (N, C)
        P  = self._softmax(Z)               # (N, C)
        Y  = np.zeros_like(P)               # one-hot: (N, C)
        Y[np.arange(N), y_b] = 1.0
        dZ = (P - Y) / N                    # (N, C)
        dW = X_b.T @ dZ + self.l2_reg * self.W   # (F, C)
        db = np.sum(dZ, axis=0)             # (C,)
        return dW, db, P

    def fit(self, X_train, y_train, X_val=None, y_val=None):
        """Train using mini-batch SGD."""
        rng = np.random.RandomState(self.seed)
        N, F = X_train.shape
        C = self.num_classes

        # Xavier initialization
        scale = math.sqrt(2.0 / (F + C))
        self.W = rng.randn(F, C).astype(np.float32) * scale
        self.b = np.zeros(C, dtype=np.float32)

        print(f'Training: {N} samples, {F} features, {C} classes')
        print(f'LR={self.learning_rate}, epochs={self.num_epochs}, batch={self.batch_size}')
        print('-' * 70)

        for epoch in range(self.num_epochs):
            idx = rng.permutation(N)
            Xs, ys = X_train[idx], y_train[idx]
            epoch_loss, nb = 0.0, 0

            for start in range(0, N, self.batch_size):
                end = min(start + self.batch_size, N)
                dW, db, P = self._gradients(Xs[start:end], ys[start:end])
                self.W -= self.learning_rate * dW
                self.b -= self.learning_rate * db
                epoch_loss += self._loss(P, ys[start:end])
                nb += 1

            avg_loss = epoch_loss / nb
            self.train_losses.append(avg_loss)

            val_info = ''
            if X_val is not None:
                vp = self.predict_proba(X_val)
                vl = self._loss(vp, y_val)
                va = float(np.mean(np.argmax(vp, axis=1) == y_val))
                self.val_losses.append(vl)
                val_info = f' | Val Loss: {vl:.4f} | Val Acc: {va:.4f}'

            print(f'Epoch {epoch+1:3d}/{self.num_epochs} | Train Loss: {avg_loss:.4f}{val_info}')

        print('-' * 70)
        print('Training complete.')
        return self

    def predict_proba(self, X):
        """Return softmax probabilities. X: (N,F) -> (N,C)"""
        return self._softmax(X @ self.W + self.b)

    def predict(self, X):
        """Return predicted class indices. X: (N,F) -> (N,)"""
        return np.argmax(self.predict_proba(X), axis=1)


print('CustomLogisticRegression defined.')


CustomLogisticRegression defined.


## Step 11: Train the Model

In [11]:
clf = CustomLogisticRegression(
    num_classes   = NUM_CLASSES,
    learning_rate = 0.5,
    num_epochs    = 30,
    batch_size    = 256,
    l2_reg        = 1e-4,
    seed          = SEED,
)
clf.fit(X_train, y_train, X_val=X_val, y_val=y_val)


Training: 18032 samples, 50000 features, 23 classes
LR=0.5, epochs=30, batch=256
----------------------------------------------------------------------
Epoch   1/30 | Train Loss: 0.7890 | Val Loss: 1.2697 | Val Acc: 0.9135
Epoch   2/30 | Train Loss: 0.2402 | Val Loss: 0.4697 | Val Acc: 0.9503
Epoch   3/30 | Train Loss: 0.0831 | Val Loss: 0.4601 | Val Acc: 0.9490
Epoch   4/30 | Train Loss: 0.0477 | Val Loss: 0.4936 | Val Acc: 0.9494
Epoch   5/30 | Train Loss: 0.0345 | Val Loss: 0.5295 | Val Acc: 0.9481
Epoch   6/30 | Train Loss: 0.0280 | Val Loss: 0.4805 | Val Acc: 0.9472
Epoch   7/30 | Train Loss: 0.0232 | Val Loss: 0.4861 | Val Acc: 0.9454
Epoch   8/30 | Train Loss: 0.0215 | Val Loss: 0.4840 | Val Acc: 0.9445
Epoch   9/30 | Train Loss: 0.0207 | Val Loss: 0.4823 | Val Acc: 0.9450
Epoch  10/30 | Train Loss: 0.0200 | Val Loss: 0.4812 | Val Acc: 0.9454
Epoch  11/30 | Train Loss: 0.0195 | Val Loss: 0.4800 | Val Acc: 0.9463
Epoch  12/30 | Train Loss: 0.0192 | Val Loss: 0.4788 | Val Acc: 0.9

## Step 12: Custom Macro-F1 Evaluation

**No sklearn used.**

For each class c:
- TP_c = both true and predicted == c
- FP_c = predicted == c but true != c
- FN_c = true == c but predicted != c
- Precision_c = TP_c / (TP_c + FP_c)
- Recall_c = TP_c / (TP_c + FN_c)
- F1_c = 2 * P_c * R_c / (P_c + R_c)

Macro-F1 = mean(F1_c for all c)

In [12]:
def compute_macro_f1(y_true, y_pred, num_classes):
    """
    Compute Macro-F1 score without sklearn.
    Returns: (macro_f1, per_class_metrics_dict)
    """
    y_true = list(y_true)
    y_pred = list(y_pred)
    assert len(y_true) == len(y_pred)

    TP      = [0] * num_classes
    FP      = [0] * num_classes
    FN      = [0] * num_classes
    support = [0] * num_classes

    for t, p in zip(y_true, y_pred):
        support[t] += 1
        if t == p:
            TP[t] += 1
        else:
            FN[t] += 1
            FP[p] += 1

    per_class = {}
    f1_scores = []
    for c in range(num_classes):
        prec = TP[c] / (TP[c] + FP[c]) if (TP[c] + FP[c]) > 0 else 0.0
        rec  = TP[c] / (TP[c] + FN[c]) if (TP[c] + FN[c]) > 0 else 0.0
        f1   = 2*prec*rec / (prec+rec)  if (prec+rec) > 0       else 0.0
        per_class[c] = {'precision': prec, 'recall': rec, 'f1': f1, 'support': support[c]}
        f1_scores.append(f1)

    macro_f1 = sum(f1_scores) / num_classes
    return macro_f1, per_class


def compute_accuracy(y_true, y_pred):
    """Simple accuracy (no sklearn)."""
    return sum(t == p for t, p in zip(y_true, y_pred)) / len(y_true)


print('Evaluation functions defined.')


Evaluation functions defined.


## Step 13: Evaluate on Validation and Test Sets

In [13]:
# ----- Validation -----
y_val_pred = clf.predict(X_val)
val_acc    = compute_accuracy(y_val, y_val_pred)
val_mf1, val_pc = compute_macro_f1(y_val, y_val_pred, NUM_CLASSES)

print('VALIDATION RESULTS')
print(f'  Accuracy : {val_acc:.4f}')
print(f'  Macro-F1 : {val_mf1:.4f}')
print()
print(f"{'Class':<20} {'Precision':>10} {'Recall':>10} {'F1':>10} {'Support':>10}")
print('-' * 65)
for c, m in val_pc.items():
    print(f"{idx2label[c]:<20} {m['precision']:>10.4f} {m['recall']:>10.4f} {m['f1']:>10.4f} {m['support']:>10}")


VALIDATION RESULTS
  Accuracy : 0.9472
  Macro-F1 : 0.9474

Class                 Precision     Recall         F1    Support
-----------------------------------------------------------------
asm_Beng                 1.0000     0.9898     0.9949         98
ben_Beng                 0.9899     1.0000     0.9949         98
brx_Deva                 0.9310     0.8265     0.8757         98
doi_Deva                 0.9444     0.8673     0.9043         98
gom_Deva                 0.7128     0.6837     0.6979         98
guj_Gujr                 1.0000     1.0000     1.0000         98
hin_Deva                 0.7946     0.9082     0.8476         98
kan_Knda                 1.0000     1.0000     1.0000         98
kas_Arab                 0.9897     0.9796     0.9846         98
khasi                    0.9800     1.0000     0.9899         98
mai_Deva                 0.9278     0.9184     0.9231         98
mal_Mlym                 1.0000     1.0000     1.0000         98
mar_Deva                 0.71

In [14]:
# ----- Test -----
y_test_pred = clf.predict(X_test)
test_acc    = compute_accuracy(y_test, y_test_pred)
test_mf1, test_pc = compute_macro_f1(y_test, y_test_pred, NUM_CLASSES)

print('TEST RESULTS')
print(f'  Accuracy : {test_acc:.4f}')
print(f'  Macro-F1 : {test_mf1:.4f}')
print()
print(f"{'Class':<20} {'Precision':>10} {'Recall':>10} {'F1':>10} {'Support':>10}")
print('-' * 65)
for c, m in test_pc.items():
    print(f"{idx2label[c]:<20} {m['precision']:>10.4f} {m['recall']:>10.4f} {m['f1']:>10.4f} {m['support']:>10}")


TEST RESULTS
  Accuracy : 0.9416
  Macro-F1 : 0.9419

Class                 Precision     Recall         F1    Support
-----------------------------------------------------------------
asm_Beng                 0.9796     0.9697     0.9746         99
ben_Beng                 0.9898     0.9798     0.9848         99
brx_Deva                 0.9355     0.8788     0.9062         99
doi_Deva                 0.9149     0.8687     0.8912         99
gom_Deva                 0.6449     0.6970     0.6699         99
guj_Gujr                 0.9900     1.0000     0.9950         99
hin_Deva                 0.8462     0.8889     0.8670         99
kan_Knda                 0.9802     1.0000     0.9900         99
kas_Arab                 0.9900     1.0000     0.9950         99
khasi                    0.9800     0.9899     0.9849         99
mai_Deva                 0.8932     0.9293     0.9109         99
mal_Mlym                 1.0000     1.0000     1.0000         99
mar_Deva                 0.7021    

## Step 14: Confusion Matrix (Custom, No sklearn)

In [15]:
def compute_confusion_matrix(y_true, y_pred, num_classes):
    """
    Confusion matrix (no sklearn).
    CM[i][j] = number of samples where true_label=i and predicted_label=j.
    """
    cm = [[0]*num_classes for _ in range(num_classes)]
    for t, p in zip(y_true, y_pred):
        cm[t][p] += 1
    return cm


cm = compute_confusion_matrix(y_test, y_test_pred, NUM_CLASSES)
short = [l[:6] for l in unique_labels]

header = f"{'':12}" + ''.join(f"{l:>7}" for l in short)
print('Confusion Matrix (Test) - rows=True, cols=Predicted')
print(header)
print('-' * len(header))
for i, row in enumerate(cm):
    print(f"{short[i]:<12}" + ''.join(f'{v:>7}' for v in row))


Confusion Matrix (Test) - rows=True, cols=Predicted
             asm_Be ben_Be brx_De doi_De gom_De guj_Gu hin_De kan_Kn kas_Ar  khasi mai_De mal_Ml mar_De mni_Mt npi_De ory_Or pan_Gu san_De santha snd_De tam_Ta tel_Te urd_Ar
-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------
asm_Be           96      1      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      2      0      0      0      0      0
ben_Be            2     97      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0      0
brx_De            0      0     87      1      1      0      8      0      0      0      1      0      0      0      0      0      0      1      0      0      0      0      0
doi_De            0      0      2     86      2      0      5      0      0   

## Step 15: Final Summary

In [16]:
print('=' * 60)
print('FINAL SUMMARY')
print('=' * 60)
total_sents = sum(len(v) for v in dataset_sentences.values())
print(f'Languages       : {NUM_CLASSES}')
print(f'Total sentences : {total_sents}')
print(f'Train/Val/Test  : {len(train_texts)}/{len(val_texts)}/{len(test_texts)}')
print(f'Vocabulary size : {len(vectorizer.vocabulary_)}')
print(f'LR / Epochs     : {clf.learning_rate} / {clf.num_epochs}')
print(f'L2 regularization: {clf.l2_reg}')
print()
print(f'Val  Accuracy   : {val_acc:.4f}  ({val_acc*100:.2f}%)')
print(f'Val  Macro-F1   : {val_mf1:.4f}')
print(f'Test Accuracy   : {test_acc:.4f}  ({test_acc*100:.2f}%)')
print(f'Test Macro-F1   : {test_mf1:.4f}')
print('=' * 60)


FINAL SUMMARY
Languages       : 23
Total sentences : 22563
Train/Val/Test  : 18032/2254/2277
Vocabulary size : 50000
LR / Epochs     : 0.5 / 30
L2 regularization: 0.0001

Val  Accuracy   : 0.9472  (94.72%)
Val  Macro-F1   : 0.9474
Test Accuracy   : 0.9416  (94.16%)
Test Macro-F1   : 0.9419
